In [ ]:
import glob
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from datetime import datetime
pd.set_option('display.max_columns', None)

events = pd.read_parquet("../data/gdelt/events/6_final/events_dataset_v2.parquet")
events = events[events['period'] <= '202406']
print(f"Loaded {len(events)} rows")

In [ ]:
# =============================================================================
# 1. Helper functions
# =============================================================================

def safe_len(x):
    if hasattr(x, '__len__'):
        return len(x)
    return 0

def safe_list_sum(lst):
    if lst is None:
        return 0
    if isinstance(lst, (list, np.ndarray)):
        vals = [v for v in lst if isinstance(v, (int, float, np.integer, np.floating)) and not (isinstance(v, float) and np.isnan(v))]
        return sum(vals) if vals else 0
    return 0

def period_to_date(period_str):
    return datetime(int(period_str[:4]), int(period_str[4:6]), 15)

# Pre-compute article/event counts
events['_n_articles'] = events['clean_text_list'].apply(safe_len)
events['_n_events'] = events['EventCode'].apply(safe_len)
print("Helper functions defined, article/event counts computed.")

In [ ]:
# =============================================================================
# 2. Identify crisis cases: IPC jumps between consecutive assessments
# =============================================================================

df = events[['ADMIN0', 'ADMIN1', 'ADMIN2', 'CS_score', 'period']].copy()
df = df[df['CS_score'].between(1, 5)].copy()
df['CS_score'] = df['CS_score'].astype(int)
df = df.sort_values(['ADMIN0', 'ADMIN1', 'ADMIN2', 'period']).reset_index(drop=True)

df['prev_CS'] = df.groupby(['ADMIN0', 'ADMIN1', 'ADMIN2'])['CS_score'].shift(1)
df['prev_period'] = df.groupby(['ADMIN0', 'ADMIN1', 'ADMIN2'])['period'].shift(1)
df['jump'] = df['CS_score'] - df['prev_CS']

# Any worsening (jump >= 1) in recent periods (2022+) with GDELT coverage
any_worsening = df[(df['jump'] >= 1) & (df['period'] >= '202201')].copy()

candidates = []
for _, row in any_worsening.iterrows():
    a0, a1, a2 = row['ADMIN0'], row['ADMIN1'], row['ADMIN2']
    crisis_period = row['period']
    
    area_data = events[
        (events['ADMIN0'] == a0) & 
        (events['ADMIN1'] == a1) & 
        (events['ADMIN2'] == a2)
    ]
    
    all_periods = sorted(area_data['period'].unique())
    if crisis_period not in all_periods:
        continue
    crisis_idx = all_periods.index(crisis_period)
    if crisis_idx < 1:
        continue
    
    lookback_periods = all_periods[max(0, crisis_idx - 6):crisis_idx]
    pre_crisis = area_data[area_data['period'].isin(lookback_periods)]
    total_articles = pre_crisis['_n_articles'].sum()
    total_events = pre_crisis['_n_events'].sum()
    
    candidates.append({
        'ADMIN0': a0, 'ADMIN1': a1, 'ADMIN2': a2,
        'prev_period': row['prev_period'], 'prev_CS': int(row['prev_CS']),
        'crisis_period': crisis_period, 'crisis_CS': int(row['CS_score']),
        'jump': int(row['jump']),
        'articles_pre_6m': total_articles,
        'events_pre_6m': total_events,
    })

candidates_df = pd.DataFrame(candidates)
good = candidates_df[candidates_df['events_pre_6m'] >= 100].sort_values(
    ['jump', 'events_pre_6m'], ascending=[False, False]
)

print(f"Recent worsenings (2022+) with >=100 events in prior 6 months: {len(good)}")

for country in ['sudan', 'ethiopia', 'somalia', 'nigeria', 'niger']:
    subset = good[good['ADMIN0'] == country]
    if len(subset) > 0:
        print(f"\n{'='*70}")
        print(f"  {country.upper()} ({len(subset)} areas)")
        print(f"{'='*70}")
        print(subset.head(10).to_string(index=False))

In [ ]:
# =============================================================================
# 3. Check IPC assessment periods for selected admin2 areas
# =============================================================================

case_studies = [
    ('sudan', 'khartoum', 'khartoum'),
    ('ethiopia', 'tigray', 'south tigray'),
    ('mozambique', 'sofala', 'dondo'),
]

for a0, a1, a2 in case_studies:
    area_ipc = events[
        (events['ADMIN0'] == a0) &
        (events['ADMIN1'] == a1) &
        (events['ADMIN2'] == a2) &
        (events['CS_score'].between(1, 5))
    ].sort_values('period')
    
    print(f"\n{'='*50}")
    print(f"{a0.title()} / {a1.title()} / {a2.title()} — {len(area_ipc)} IPC assessments")
    print(f"{'='*50}")
    
    prev_period = None
    for _, row in area_ipc.iterrows():
        gap = ""
        if prev_period is not None:
            prev_date = period_to_date(prev_period)
            curr_date = period_to_date(row['period'])
            months_gap = (curr_date.year - prev_date.year) * 12 + (curr_date.month - prev_date.month)
            if months_gap > 5:
                gap = f"  ← GAP: {months_gap} months"
        print(f"  {row['period']}: IPC {int(row['CS_score'])}{gap}")
        prev_period = row['period']

In [ ]:
# =============================================================================
# 4. Admin2-level timeline plots
# =============================================================================

def plot_admin2_timeline(admin0, admin1, admin2, start_period='202001'):
    """Plot GDELT signals vs IPC for a single admin2 area."""
    area = events[
        (events['ADMIN0'] == admin0) &
        (events['ADMIN1'] == admin1) &
        (events['ADMIN2'] == admin2) &
        (events['period'] >= start_period)
    ].copy()
    
    if len(area) == 0:
        print(f"No data for {admin0}/{admin1}/{admin2}")
        return
    
    area['n_events'] = area['EventCode'].apply(safe_len)
    area['n_articles'] = area['clean_text_list'].apply(safe_len)
    area['fatalities_freq'] = area['fatalities_freq_list'].apply(safe_list_sum)
    area['displaced_freq'] = area['displaced_freq_list'].apply(safe_list_sum)
    area['food_insec_freq'] = area['food_insecurity_freq_list'].apply(safe_list_sum)
    area['weather_freq'] = area['weather_freq_list'].apply(safe_list_sum)
    area['fatalities_count'] = area['fatalities_count_list'].apply(safe_list_sum)
    area['displaced_count'] = area['displaced_count_list'].apply(safe_list_sum)
    
    area['date'] = area['period'].apply(period_to_date)
    area = area.sort_values('date')
    
    ipc = area[area['CS_score'].between(1, 5)][['date', 'period', 'CS_score']].copy()
    ipc['CS_score'] = ipc['CS_score'].astype(int)
    
    ipc_colors = {1: '#c6dbef', 2: '#fdd49e', 3: '#fc8d59', 4: '#e34a33', 5: '#b30000'}
    
    fig, axes = plt.subplots(4, 1, figsize=(14, 12), sharex=True,
                              gridspec_kw={'height_ratios': [1.2, 1, 1, 1]})
    fig.suptitle(
        f"GDELT Early Warning Signals vs IPC Assessment\n"
        f"{admin0.title()} — {admin1.title()} — {admin2.title()}",
        fontsize=14, fontweight='bold', y=0.98)
    
    # Panel 1: Event count + IPC phase
    ax1 = axes[0]
    ax1.bar(area['date'], area['n_events'], width=25, color='steelblue', alpha=0.7, label='GDELT events')
    ax1.set_ylabel('GDELT event count', color='steelblue')
    ax1.tick_params(axis='y', labelcolor='steelblue')
    
    ax1_twin = ax1.twinx()
    for _, row in ipc.iterrows():
        color = ipc_colors.get(row['CS_score'], 'gray')
        ax1_twin.scatter(row['date'], row['CS_score'], c=color, s=120,
                        edgecolors='black', linewidth=1.5, zorder=5)
        ax1_twin.annotate(f"IPC {int(row['CS_score'])}", xy=(row['date'], row['CS_score']),
                         xytext=(0, 12), textcoords='offset points', fontsize=9,
                         ha='center', fontweight='bold',
                         bbox=dict(boxstyle='round,pad=0.2', facecolor='white', alpha=0.8))
    ax1_twin.plot(ipc['date'], ipc['CS_score'], 'k--', alpha=0.4, linewidth=1)
    ax1_twin.set_ylabel('IPC Phase', color='black')
    ax1_twin.set_ylim(0.5, 5.5)
    ax1_twin.set_yticks([1, 2, 3, 4, 5])
    ax1.set_title('Media Coverage & IPC Phase')
    ax1.legend(loc='upper left')
    
    # Panel 2: Scraped articles (NLP-processed)
    ax2 = axes[1]
    ax2.bar(area['date'], area['n_articles'], width=25, color='darkorange', alpha=0.7, label='Scraped articles (NLP features)')
    ax2.set_ylabel('Scraped article count')
    ax2.set_title('NLP-Processed Articles (subset used for feature extraction)')
    ax2.legend(loc='upper left')
    
    # Panel 3: Reported counts
    ax3 = axes[2]
    ax3.bar(area['date'], area['fatalities_count'], width=20, color='red', alpha=0.6, label='Fatalities (reported)')
    ax3.bar(area['date'], area['displaced_count'], width=20, color='orange', alpha=0.6,
            bottom=area['fatalities_count'], label='Displaced (reported)')
    ax3.set_ylabel('Reported count')
    ax3.set_title('Crisis Indicators — reported fatalities & displacement')
    ax3.legend(loc='upper left')
    
    # Panel 4: Keyword frequencies
    ax4 = axes[3]
    ax4.plot(area['date'], area['fatalities_freq'], color='red', linewidth=1.5,
             marker='o', markersize=3, label='Fatalities mentions')
    ax4.plot(area['date'], area['displaced_freq'], color='orange', linewidth=1.5,
             marker='s', markersize=3, label='Displacement mentions')
    ax4.plot(area['date'], area['food_insec_freq'], color='darkgreen', linewidth=1.5,
             marker='^', markersize=3, label='Food insecurity mentions')
    ax4.plot(area['date'], area['weather_freq'], color='dodgerblue', linewidth=1.5,
             marker='d', markersize=3, label='Weather/climate mentions')
    ax4.set_ylabel('# articles mentioning')
    ax4.set_title('Keyword Frequencies')
    ax4.legend(loc='upper left')
    ax4.xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m'))
    ax4.xaxis.set_major_locator(mdates.MonthLocator(interval=3))
    plt.setp(ax4.xaxis.get_majorticklabels(), rotation=45, ha='right')
    
    crisis_markers = {
        'sudan': (datetime(2023, 4, 15), 'Civil war onset\nApr 2023', 'red'),
        'zimbabwe': (datetime(2019, 6, 1), 'Economic crisis\n+ drought 2019', 'purple'),
        'ethiopia': (datetime(2020, 11, 4), 'Tigray war onset\nNov 2020', 'darkred'),
        'mozambique': (datetime(2019, 3, 14), 'Cyclone Idai\nMar 2019', 'teal'),
    }
    if admin0 in crisis_markers:
        marker_date, label, color = crisis_markers[admin0]
        for ax in axes:
            ax.axvline(x=marker_date, color=color, linestyle='--', alpha=0.7, linewidth=2)
            ax.annotate(label, xy=(marker_date, ax.get_ylim()[1]*0.9),
                        fontsize=8, color=color, ha='center')
    
    plt.tight_layout()
    fname = f'crisis_timeline_{admin0}_{admin1}_{admin2}.png'.replace(' ', '_')
    plt.savefig(fname, dpi=150, bbox_inches='tight')
    plt.show()
    print(f"Saved: {fname}")

# Generate plots for 4 case studies (diverse countries and crisis drivers)
plot_admin2_timeline('sudan', 'khartoum', 'khartoum', start_period='202001')        # Civil war
plot_admin2_timeline('ethiopia', 'tigray', 'south tigray', start_period='201901')   # Regional conflict
plot_admin2_timeline('mozambique', 'sofala', 'dondo', start_period='201801')        # Cyclone Idai (weather)